In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(r"D:\Khushi\Cleansing_Data\us\texas\tarrant\transaction_log\Cleaned_Data\us_tx_48439_data_prop_sales_silver_20260218.csv")
df

In [ ]:
df.info()

In [ ]:
df['book/page'] = df.apply(
    lambda r: f"{r.deed_volume}/{r.deed_page}"
    if pd.notna(r.deed_volume) and pd.notna(r.deed_page) 
    else pd.NA,
    axis=1
)
df

In [ ]:
df2 = pd.read_csv(r"D:\Khushi\Cleansing_Data\us\texas\tarrant\transaction_log\Raw_Data\us_tx_48439_data_trxlog_main_bronze_20260218.csv",dtype={'input_page':str,'input_book':str})
df2

In [ ]:
col = df2.columns.difference(['input_instrument','input_book','input_page','deed_document_present', 'image_save', 'page_save'])
mask = df2[col].isna().all(axis=1)
df2[mask]

In [ ]:
removed_df = df2.loc[mask]
df2 = df2.loc[~mask].reset_index(drop=True)

In [ ]:
col = df2.columns.difference(['input_instrument','input_book','input_page','deed_document_present', 'image_save', 'page_save'])
mask = df2[col].isna().all(axis=1)
df2[mask]

In [ ]:
df2['book_page'] = df2.apply(
    lambda r: f"{r.input_book}/{r.input_page}"
    if pd.notna(r.input_book) and pd.notna(r.input_page) 
    else pd.NA,
    axis=1
)
df2

In [ ]:
df[df['book/page']=='0007496/0001218']

In [ ]:
df2

In [ ]:
df2.columns

In [ ]:
df2 = df2.rename(columns={'instrument':'instrument_number'})

In [ ]:
df2

In [ ]:
df.columns

In [ ]:
df2[df2['book_page']=='0011686/0000180']

In [ ]:
# import re
# df['book/page'] = df['book/page'].apply(
#     lambda x: str(x).replace('.0', '') if pd.notna(x) else x
# )

# df['book/page'] = df['book/page'].apply(
#     lambda x: re.sub(r'\b0+(\d)', r'\1', x) if pd.notna(x) else x
# )


# df['book/page']

In [ ]:
# m1 = df.merge(
#     df2[['instrument_number', 'book_page','book/liber/page']],
#     how='left',
#     left_on='instrument',
#     right_on='instrument_number'
# )
# m1

In [ ]:
# Convert both to string and strip spaces
df['book/page'] = df['book/page'].astype(str).str.strip()
df2['book_page'] = df2['book_page'].astype(str).str.strip()


In [ ]:
# df['book/page'] = df['book/page'].apply(
#     lambda x: f"{str(x).split('/')[0]}/{int(float(str(x).split('/')[1])):06d}" 
#     if pd.notna(x) and '/' in str(x) else x
# )
# df['book/page']

In [ ]:
print(df['instrument'].value_counts().to_string())

In [ ]:
invalid = df[
    (df['instrument'].astype(str).str.fullmatch(r'0+')) |      # all zeros
    (df['instrument'].astype(str).str.len() < 5) |             # very short values like DC
    (df['instrument'].astype(str).str.fullmatch(r'[A-Za-z]+')) # only letters
]

invalid


In [ ]:
print(invalid['instrument'].value_counts().to_string())

## Found invalid instrument_number  like '00000000000000' , 'DC' ,'D/C''452' so remove this in the joined sales file 

In [ ]:
df = df[
    ~(
        df['instrument'].astype(str).str.fullmatch(r'0+') |      # all zeros
        (df['instrument'].astype(str).str.len() < 5) |           # very short
        df['instrument'].astype(str).str.fullmatch(r'[A-Za-z]+') # only letters
    )
]
df

In [ ]:
df2['book_page'].duplicated().sum() == 0


In [ ]:
m1 = df.merge(
    df2[['instrument_number', 'book_page', 'book/liber/page']],
    how='left',
    left_on='instrument',
    right_on='instrument_number'
)
m1

In [ ]:
df2_unique = df2.drop_duplicates('book_page')

m2 = m1[mask].merge(
    df2_unique[['instrument_number', 'book_page', 'book/liber/page']],
    how='left',
    left_on='book/page',
    right_on='book_page'
)
m2

In [ ]:
joined_sales_file3 = pd.concat([
    m1[~mask],
    m2
], ignore_index=True)
joined_sales_file3

In [ ]:
df2[df2['book_page']=='0011686/0000180']

In [ ]:
m2[m2['book_page']=='0011686/0000180']

In [ ]:
joined_sales_file3['instrument'].nunique()


In [ ]:
joined_sales_file3['instrument_number'].nunique()


In [ ]:
joined_sales_file3['book_page'].nunique()


In [ ]:
df['book/page'].nunique()

In [ ]:
df['instrument'].nunique()

In [ ]:
2111166 - 2028160

In [ ]:
joined_sales_file3['book/page'].nunique()

In [ ]:
joined_sales_file3['book/liber/page'].nunique()

In [ ]:
joined_sales_file3['book_page'].nunique()

In [ ]:
27415 + 194426

In [ ]:
joined_sales_file3['instrument_number'].nunique()

In [ ]:
missing_df = joined_sales_file3[
    ~joined_sales_file3['book/page'].isin(joined_sales_file3['book_page'])
]
missing_df

In [ ]:
df2[df2['book_page']=='0007496/0001208']

In [ ]:
df2[df2['page_save']=='0007496_1208.json']

In [ ]:
missing_df['book/page'].nunique()